# EA-TTM — Extreme Target and Adapter Architecture

**Project:** Extreme-Aware Tiny Time Mixers (EA-TTM) for Retail Demand Forecasting under Regime Shifts  
**Phase:** 07 — Extreme-regime target and EA-TTM architecture  
**Scope:** Define the training target for the Extreme Gate, construct the complete shock-aware residual adapter, verify its tensor contract, and save an auditable initialization state.

This phase performs **no EA-TTM optimization** and does **not** evaluate the locked final-test interval.

## Frozen base forecaster

The head-probed TTM from Phase 04 remains fixed. For horizon $h$:

$$
\hat y_{\mathrm{HP},h}
$$

is the base forecast.

Phase 05 provides the horizon-aligned Slim Decoder representation:

$$
\mathbf d_h
\in
\mathbb{R}^{128}.
$$

Phase 06 provides:

$$
\mathbf z_t
\in
\mathbb{R}^{5}
$$

and:

$$
\mathbf c_h
\in
\mathbb{R}^{8}.
$$

The Retail Shock Encoder therefore receives:

$$
\mathbf u_h
=
[
\mathbf z_t;
\mathbf d_h;
\mathbf c_h
]
\in
\mathbb{R}^{141}.
$$

## Extreme-regime definition

The gate must identify unusually large positive **or** negative demand deviations relative to the recent local demand regime.

For each forecasting sample, let the preceding 28-day demand window be:

$$
\mathcal R_t
=
\{y_{t-27},\ldots,y_t\}.
$$

Define the recent center:

$$
m_t
=
\operatorname{median}(\mathcal R_t),
$$

and robust scale:

$$
s_t
=
1.4826
\operatorname{median}
\left(
|y_i-m_t|
\right).
$$

Because M5 contains many zero-demand windows, the effective scale uses a one-unit floor:

$$
\tilde s_t
=
\max(s_t,1).
$$

For future horizon $h$, the robust shock score is:

$$
q_{t,h}
=
\frac{
|y_{t+h}-m_t|
}{
\tilde s_t
}.
$$

The extreme threshold is **not manually guessed**. It is estimated once from the training windows only:

$$
\tau
=
Q_{0.95}
\left(
q_{t,h}^{\mathrm{train}}
\right).
$$

The binary gate target is then:

$$
e_{t,h}
=
\mathbb{1}
\left(
q_{t,h}>\tau
\right).
$$

The same fixed training-derived threshold $\tau$ is applied to validation.

## Residual correction target

The Correction Head does not replace the HP-TTM forecast. It learns a signed residual:

$$
r^{*}_{t,h}
=
y_{t+h}
-
\hat y_{\mathrm{HP},t,h}.
$$

The final EA-TTM composition is:

$$
\hat y_{\mathrm{EA},t,h}
=
\hat y_{\mathrm{HP},t,h}
+
g_{t,h}
\Delta_{t,h},
$$

where:

$$
g_{t,h}
=
\sigma(
W_g\mathbf r_{t,h}+b_g
)
$$

and:

$$
\Delta_{t,h}
=
f_{\Delta}
(
\mathbf r_{t,h}
).
$$

## Adapter parameterization

The Retail Shock Encoder is deliberately small:

$$
141
\rightarrow
16.
$$

The architecture is:

- `Linear(141, 16)`
- `GELU`
- `LayerNorm(16)`

The Extreme Gate is:

$$
16
\rightarrow
1
\rightarrow
\sigma(\cdot).
$$

The Correction Head is:

$$
16
\rightarrow
8
\rightarrow
1,
$$

with GELU between its two linear layers.

No activation constrains the final correction because $\Delta_h$ must be able to take positive or negative values.

## Stable initialization

The adapter must begin as an exact identity extension of HP-TTM.

The final Correction Head layer is initialized to zero:

$$
W_{\Delta,2}=0,
\qquad
b_{\Delta,2}=0.
$$

Therefore:

$$
\Delta_h=0
$$

at initialization, giving:

$$
\hat y_{\mathrm{EA},h}
=
\hat y_{\mathrm{HP},h}.
$$

The gate weight is initialized to zero and its bias to the log-odds of the training extreme prevalence:

$$
b_g
=
\log
\left(
\frac{\pi_e}{1-\pi_e}
\right),
$$

where $\pi_e$ is the observed training extreme rate.

Hence the initial gate probability equals the empirical training prior while the zero correction guarantees that the adapter cannot degrade the base forecast before learning begins.

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import os
import random
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from torch import nn

## Project preflight

In [2]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/EA_TTM_RETAIL")

PROCESSED_M5_DIR = PROJECT_ROOT / "data" / "processed" / "m5"
METADATA_DIR = PROJECT_ROOT / "metadata"
RESULTS_TABLE_DIR = PROJECT_ROOT / "results" / "tables"

PHASE05_CACHE_DIR = PROJECT_ROOT / "runs" / "05_decoder_crossfeed" / "cache"
PHASE06_FEATURE_DIR = PROJECT_ROOT / "runs" / "06_retail_features" / "features"

RUN_DIR = PROJECT_ROOT / "runs" / "07_ea_ttm_architecture"
TARGET_DIR = RUN_DIR / "targets"
MODEL_DIR = PROJECT_ROOT / "models" / "ea_ttm_adapter_initial"

for directory in (
    RESULTS_TABLE_DIR,
    RUN_DIR,
    TARGET_DIR,
    MODEL_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

required_artifacts = {
    "split_manifest": METADATA_DIR / "02_SPLIT_MANIFEST.json",
    "phase04_summary": METADATA_DIR / "04_RUN_SUMMARY.json",
    "phase05_summary": METADATA_DIR / "05_RUN_SUMMARY.json",
    "phase05_cache_manifest": METADATA_DIR / "05_CACHE_MANIFEST.json",
    "decoder_alignment_weights": METADATA_DIR / "05_DECODER_ALIGNMENT_WEIGHTS.npz",
    "phase06_feature_contract": METADATA_DIR / "06_FEATURE_CONTRACT.json",
    "phase06_summary": METADATA_DIR / "06_RUN_SUMMARY.json",
    "processed_sales": PROCESSED_M5_DIR / "sales_train_evaluation.parquet",
    "train_decoder_hidden": PHASE05_CACHE_DIR / "train_decoder_hidden.npy",
    "train_base_forecast": PHASE05_CACHE_DIR / "train_base_forecast.npy",
    "validation_decoder_hidden": PHASE05_CACHE_DIR / "validation_decoder_hidden.npy",
    "validation_base_forecast": PHASE05_CACHE_DIR / "validation_base_forecast.npy",
    "train_recent_features": PHASE06_FEATURE_DIR / "train_recent_demand_features.npy",
    "validation_recent_features": PHASE06_FEATURE_DIR / "validation_recent_demand_features.npy",
    "train_future_covariates": PHASE06_FEATURE_DIR / "train_future_covariates.npy",
    "validation_future_covariates": PHASE06_FEATURE_DIR / "validation_future_covariates.npy",
}

artifact_preflight = {
    name: {
        "path": str(path),
        "exists": path.exists(),
    }
    for name, path in required_artifacts.items()
}

missing_artifacts = [
    item["path"]
    for item in artifact_preflight.values()
    if not item["exists"]
]

assert not missing_artifacts, (
    "Phase 07 prerequisites are incomplete: "
    + "; ".join(missing_artifacts)
)

Mounted at /content/drive


## Recover locked project contracts

In [3]:
split_manifest = json.loads(
    (METADATA_DIR / "02_SPLIT_MANIFEST.json").read_text(
        encoding="utf-8"
    )
)

phase04_summary = json.loads(
    (METADATA_DIR / "04_RUN_SUMMARY.json").read_text(
        encoding="utf-8"
    )
)

phase05_summary = json.loads(
    (METADATA_DIR / "05_RUN_SUMMARY.json").read_text(
        encoding="utf-8"
    )
)

cache_manifest = json.loads(
    (METADATA_DIR / "05_CACHE_MANIFEST.json").read_text(
        encoding="utf-8"
    )
)

feature_contract = json.loads(
    (METADATA_DIR / "06_FEATURE_CONTRACT.json").read_text(
        encoding="utf-8"
    )
)

phase06_summary = json.loads(
    (METADATA_DIR / "06_RUN_SUMMARY.json").read_text(
        encoding="utf-8"
    )
)

DATASET_SHA256 = phase06_summary["dataset_sha256"]
SPLIT_SHA256 = phase06_summary["split_sha256"]
FEATURE_CONTRACT_SHA256 = phase06_summary["feature_contract_sha256"]
CROSSFEED_CONTRACT_SHA256 = phase05_summary["crossfeed_contract_sha256"]
CACHE_CONTRACT_SHA256 = phase05_summary["cache_contract_sha256"]

CONTEXT_LENGTH = int(split_manifest["context_length"])
FORECAST_HORIZON = int(split_manifest["forecast_horizon"])
TRAIN_ORIGINS = [int(x) for x in split_manifest["train_origins"]]
VALIDATION_ORIGIN = int(split_manifest["validation_origin"])

TRAINING_SAMPLE_COUNT = int(
    cache_manifest["train_decoder_hidden"]["shape"][0]
)

VALIDATION_SAMPLE_COUNT = int(
    cache_manifest["validation_decoder_hidden"]["shape"][0]
)

assert DATASET_SHA256 == phase05_summary["dataset_sha256"]
assert SPLIT_SHA256 == split_manifest["split_sha256"]
assert FEATURE_CONTRACT_SHA256 == feature_contract["feature_contract_sha256"]
assert CONTEXT_LENGTH == 512
assert FORECAST_HORIZON == 28
assert TRAIN_ORIGINS == [512, 960, 1409, 1857]
assert VALIDATION_ORIGIN == 1885
assert TRAINING_SAMPLE_COUNT == 121_960
assert VALIDATION_SAMPLE_COUNT == 30_490
assert feature_contract["retail_shock_encoder_input_dimension"] == 141
assert split_manifest["locked_test_interval"] == ["d_1914", "d_1941"]

## Reproducibility state

In [4]:
BASE_SEED = 2026

def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(
        True,
        warn_only=True,
    )

set_global_seed(BASE_SEED)

## Load development targets and reconstruct sample order

Only demand through the validation target end (`d_1913`) is loaded.

In [5]:
development_columns = [
    f"d_{day}"
    for day in range(
        1,
        VALIDATION_ORIGIN
        + FORECAST_HORIZON
        + 1,
    )
]

development_frame = pd.read_parquet(
    PROCESSED_M5_DIR / "sales_train_evaluation.parquet",
    columns=development_columns,
)

development_values = development_frame.to_numpy(
    dtype=np.float32,
    copy=True,
)

assert development_values.shape == (
    30_490,
    1913,
)

training_targets = np.empty(
    (
        TRAINING_SAMPLE_COUNT,
        FORECAST_HORIZON,
    ),
    dtype=np.float32,
)

training_recent_median = np.empty(
    TRAINING_SAMPLE_COUNT,
    dtype=np.float32,
)

training_recent_scale = np.empty(
    TRAINING_SAMPLE_COUNT,
    dtype=np.float32,
)

for origin_index, origin in enumerate(TRAIN_ORIGINS):
    destination_start = (
        origin_index
        * VALIDATION_SAMPLE_COUNT
    )

    destination_end = (
        destination_start
        + VALIDATION_SAMPLE_COUNT
    )

    recent_window = development_values[
        :,
        origin - 28:origin,
    ].astype(
        np.float64,
        copy=False,
    )

    median = np.median(
        recent_window,
        axis=1,
    )

    mad = np.median(
        np.abs(
            recent_window
            - median[:, None]
        ),
        axis=1,
    )

    training_recent_median[
        destination_start:destination_end
    ] = median.astype(
        np.float32
    )

    training_recent_scale[
        destination_start:destination_end
    ] = (
        1.4826
        * mad
    ).astype(
        np.float32
    )

    training_targets[
        destination_start:destination_end
    ] = development_values[
        :,
        origin:origin + FORECAST_HORIZON,
    ]

validation_recent_window = development_values[
    :,
    VALIDATION_ORIGIN - 28:VALIDATION_ORIGIN,
].astype(
    np.float64,
    copy=False,
)

validation_recent_median = np.median(
    validation_recent_window,
    axis=1,
).astype(
    np.float32
)

validation_recent_mad = np.median(
    np.abs(
        validation_recent_window
        - validation_recent_median[:, None]
    ),
    axis=1,
)

validation_recent_scale = (
    1.4826
    * validation_recent_mad
).astype(
    np.float32
)

validation_target = development_values[
    :,
    VALIDATION_ORIGIN:
    VALIDATION_ORIGIN + FORECAST_HORIZON,
]

assert training_targets.shape == (
    121_960,
    28,
)

assert validation_target.shape == (
    30_490,
    28,
)

assert "d_1914" not in development_columns

## Training-only extreme threshold

In [6]:
ROBUST_SCALE_FLOOR = 1.0
EXTREME_QUANTILE = 0.95

training_effective_scale = np.maximum(
    training_recent_scale,
    ROBUST_SCALE_FLOOR,
).astype(
    np.float32
)

training_shock_scores = (
    np.abs(
        training_targets
        - training_recent_median[:, None]
    )
    / training_effective_scale[:, None]
).astype(
    np.float32
)

extreme_threshold = float(
    np.quantile(
        training_shock_scores,
        EXTREME_QUANTILE,
    )
)

training_extreme_labels = (
    training_shock_scores
    > extreme_threshold
).astype(
    np.uint8
)

validation_effective_scale = np.maximum(
    validation_recent_scale,
    ROBUST_SCALE_FLOOR,
).astype(
    np.float32
)

validation_shock_scores = (
    np.abs(
        validation_target
        - validation_recent_median[:, None]
    )
    / validation_effective_scale[:, None]
).astype(
    np.float32
)

validation_extreme_labels = (
    validation_shock_scores
    > extreme_threshold
).astype(
    np.uint8
)

training_extreme_rate = float(
    training_extreme_labels.mean(
        dtype=np.float64
    )
)

validation_extreme_rate = float(
    validation_extreme_labels.mean(
        dtype=np.float64
    )
)

assert 0.0 < training_extreme_rate < 1.0
assert 0.0 < validation_extreme_rate < 1.0

print("Training-derived extreme threshold:", extreme_threshold)
print("Training extreme rate:", training_extreme_rate)
print("Validation extreme rate:", validation_extreme_rate)

Training-derived extreme threshold: 2.0
Training extreme rate: 0.04705260506957785
Validation extreme rate: 0.0572998172702994


## Extreme direction audit

The gate target is unsigned, while the Correction Head remains signed. Positive and negative shocks are therefore audited separately without creating separate gate classes.

In [7]:
training_signed_deviation = (
    training_targets
    - training_recent_median[:, None]
)

validation_signed_deviation = (
    validation_target
    - validation_recent_median[:, None]
)

training_positive_extreme = (
    (training_extreme_labels == 1)
    & (training_signed_deviation > 0.0)
)

training_negative_extreme = (
    (training_extreme_labels == 1)
    & (training_signed_deviation < 0.0)
)

validation_positive_extreme = (
    (validation_extreme_labels == 1)
    & (validation_signed_deviation > 0.0)
)

validation_negative_extreme = (
    (validation_extreme_labels == 1)
    & (validation_signed_deviation < 0.0)
)

training_positive_extreme_rate = float(
    training_positive_extreme.mean(
        dtype=np.float64
    )
)

training_negative_extreme_rate = float(
    training_negative_extreme.mean(
        dtype=np.float64
    )
)

validation_positive_extreme_rate = float(
    validation_positive_extreme.mean(
        dtype=np.float64
    )
)

validation_negative_extreme_rate = float(
    validation_negative_extreme.mean(
        dtype=np.float64
    )
)

## Load frozen HP-TTM caches and Phase 06 features

In [8]:
train_decoder_hidden = np.load(
    PHASE05_CACHE_DIR / "train_decoder_hidden.npy",
    mmap_mode="r",
)

train_base_forecast = np.load(
    PHASE05_CACHE_DIR / "train_base_forecast.npy",
    mmap_mode="r",
)

validation_decoder_hidden = np.load(
    PHASE05_CACHE_DIR / "validation_decoder_hidden.npy",
    mmap_mode="r",
)

validation_base_forecast = np.load(
    PHASE05_CACHE_DIR / "validation_base_forecast.npy",
    mmap_mode="r",
)

train_recent_features = np.load(
    PHASE06_FEATURE_DIR / "train_recent_demand_features.npy",
    mmap_mode="r",
)

validation_recent_features = np.load(
    PHASE06_FEATURE_DIR / "validation_recent_demand_features.npy",
    mmap_mode="r",
)

train_future_covariates = np.load(
    PHASE06_FEATURE_DIR / "train_future_covariates.npy",
    mmap_mode="r",
)

validation_future_covariates = np.load(
    PHASE06_FEATURE_DIR / "validation_future_covariates.npy",
    mmap_mode="r",
)

alignment = np.load(
    METADATA_DIR / "05_DECODER_ALIGNMENT_WEIGHTS.npz"
)

selected_weight = alignment[
    "selected_weight"
].astype(
    np.float32,
    copy=False,
)

assert train_decoder_hidden.shape == (
    121_960,
    1,
    9,
    128,
)

assert train_base_forecast.shape == (
    121_960,
    28,
)

assert validation_decoder_hidden.shape == (
    30_490,
    1,
    9,
    128,
)

assert validation_base_forecast.shape == (
    30_490,
    28,
)

assert train_recent_features.shape == (
    121_960,
    5,
)

assert train_future_covariates.shape == (
    121_960,
    28,
    8,
)

assert selected_weight.shape == (
    28,
    9,
    128,
)

## Residual targets

In [9]:
training_residual_targets = (
    training_targets
    - np.asarray(
        train_base_forecast,
        dtype=np.float32,
    )
).astype(
    np.float32
)

validation_residual_targets = (
    validation_target
    - np.asarray(
        validation_base_forecast,
        dtype=np.float32,
    )
).astype(
    np.float32
)

assert np.isfinite(
    training_residual_targets
).all()

assert np.isfinite(
    validation_residual_targets
).all()

## EA-TTM adapter implementation

In [10]:
class RetailShockAdapter(nn.Module):
    def __init__(
        self,
        input_dim: int = 141,
        encoder_width: int = 16,
        correction_hidden_width: int = 8,
        extreme_prior: float = 0.05,
    ) -> None:
        super().__init__()

        if not 0.0 < extreme_prior < 1.0:
            raise ValueError(
                "extreme_prior must lie strictly between 0 and 1."
            )

        self.input_dim = int(input_dim)
        self.encoder_width = int(encoder_width)

        self.shock_encoder = nn.Sequential(
            nn.Linear(
                input_dim,
                encoder_width,
            ),
            nn.GELU(),
            nn.LayerNorm(
                encoder_width,
            ),
        )

        self.extreme_gate = nn.Linear(
            encoder_width,
            1,
        )

        self.correction_head = nn.Sequential(
            nn.Linear(
                encoder_width,
                correction_hidden_width,
            ),
            nn.GELU(),
            nn.Linear(
                correction_hidden_width,
                1,
            ),
        )

        self._initialize_output_heads(
            extreme_prior=extreme_prior,
        )

    def _initialize_output_heads(
        self,
        extreme_prior: float,
    ) -> None:
        # Start the gate from the empirical extreme prior.
        nn.init.zeros_(
            self.extreme_gate.weight
        )

        prior_logit = math.log(
            extreme_prior
            / (1.0 - extreme_prior)
        )

        nn.init.constant_(
            self.extreme_gate.bias,
            prior_logit,
        )

        # Start as an exact identity extension of HP-TTM.
        correction_output_layer = (
            self.correction_head[-1]
        )

        nn.init.zeros_(
            correction_output_layer.weight
        )

        nn.init.zeros_(
            correction_output_layer.bias
        )

    def forward(
        self,
        recent_features: torch.Tensor,
        decoder_features: torch.Tensor,
        future_covariates: torch.Tensor,
        base_forecast: torch.Tensor,
    ) -> dict[str, torch.Tensor]:
        if recent_features.ndim != 2:
            raise ValueError(
                "recent_features must have shape [B, 5]."
            )

        if decoder_features.ndim != 3:
            raise ValueError(
                "decoder_features must have shape [B, H, 128]."
            )

        if future_covariates.ndim != 3:
            raise ValueError(
                "future_covariates must have shape [B, H, 8]."
            )

        if base_forecast.ndim != 2:
            raise ValueError(
                "base_forecast must have shape [B, H]."
            )

        batch_size, horizon, decoder_dim = (
            decoder_features.shape
        )

        if horizon != FORECAST_HORIZON:
            raise ValueError(
                f"Expected horizon={FORECAST_HORIZON}; received {horizon}."
            )

        if decoder_dim != 128:
            raise ValueError(
                f"Expected decoder_dim=128; received {decoder_dim}."
            )

        recent_expanded = (
            recent_features
            .unsqueeze(1)
            .expand(
                -1,
                horizon,
                -1,
            )
        )

        adapter_input = torch.cat(
            [
                recent_expanded,
                decoder_features,
                future_covariates,
            ],
            dim=-1,
        )

        if adapter_input.shape[-1] != self.input_dim:
            raise RuntimeError(
                "Retail Shock Encoder input dimension mismatch."
            )

        shock_representation = (
            self.shock_encoder(
                adapter_input
            )
        )

        gate_logits = (
            self.extreme_gate(
                shock_representation
            ).squeeze(-1)
        )

        gate_probability = torch.sigmoid(
            gate_logits
        )

        residual_correction = (
            self.correction_head(
                shock_representation
            ).squeeze(-1)
        )

        final_forecast = (
            base_forecast
            + gate_probability
            * residual_correction
        )

        return {
            "adapter_input": adapter_input,
            "shock_representation": shock_representation,
            "gate_logits": gate_logits,
            "gate_probability": gate_probability,
            "residual_correction": residual_correction,
            "final_forecast": final_forecast,
        }

## Architecture smoke test

The smoke test reconstructs the Phase 05 horizon-aligned decoder representation from cached raw decoder states and the frozen alignment weights.

In [11]:
SMOKE_BATCH_SIZE = 64

smoke_raw_decoder = torch.from_numpy(
    np.asarray(
        validation_decoder_hidden[:SMOKE_BATCH_SIZE],
        dtype=np.float32,
    )
)

smoke_alignment_weight = torch.from_numpy(
    selected_weight
)

smoke_decoder_features = torch.einsum(
    "bpk,hpk->bhk",
    smoke_raw_decoder[:, 0, :, :],
    smoke_alignment_weight,
)

smoke_recent_features = torch.from_numpy(
    np.asarray(
        validation_recent_features[:SMOKE_BATCH_SIZE],
        dtype=np.float32,
    )
)

smoke_future_covariates = torch.from_numpy(
    np.asarray(
        validation_future_covariates[:SMOKE_BATCH_SIZE],
        dtype=np.float32,
    )
)

smoke_base_forecast = torch.from_numpy(
    np.asarray(
        validation_base_forecast[:SMOKE_BATCH_SIZE],
        dtype=np.float32,
    )
)

adapter = RetailShockAdapter(
    input_dim=141,
    encoder_width=16,
    correction_hidden_width=8,
    extreme_prior=training_extreme_rate,
)

adapter.eval()

with torch.no_grad():
    smoke_output = adapter(
        recent_features=smoke_recent_features,
        decoder_features=smoke_decoder_features,
        future_covariates=smoke_future_covariates,
        base_forecast=smoke_base_forecast,
    )

assert smoke_output["adapter_input"].shape == (
    SMOKE_BATCH_SIZE,
    28,
    141,
)

assert smoke_output["shock_representation"].shape == (
    SMOKE_BATCH_SIZE,
    28,
    16,
)

assert smoke_output["gate_probability"].shape == (
    SMOKE_BATCH_SIZE,
    28,
)

assert smoke_output["residual_correction"].shape == (
    SMOKE_BATCH_SIZE,
    28,
)

assert smoke_output["final_forecast"].shape == (
    SMOKE_BATCH_SIZE,
    28,
)

/tmp/ipykernel_1349/1956337671.py:3: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  smoke_raw_decoder = torch.from_numpy(


## Initialization invariants

In [12]:
initial_correction_max_abs = float(
    smoke_output[
        "residual_correction"
    ].abs().max().item()
)

initial_forecast_max_abs_difference = float(
    (
        smoke_output[
            "final_forecast"
        ]
        - smoke_base_forecast
    ).abs().max().item()
)

initial_gate_mean = float(
    smoke_output[
        "gate_probability"
    ].mean().item()
)

assert initial_correction_max_abs == 0.0
assert initial_forecast_max_abs_difference == 0.0

assert np.isclose(
    initial_gate_mean,
    training_extreme_rate,
    rtol=1e-5,
    atol=1e-6,
)

print(
    "Initial correction max |Δ|:",
    initial_correction_max_abs,
)

print(
    "Initial EA-TTM vs HP-TTM max |difference|:",
    initial_forecast_max_abs_difference,
)

print(
    "Initial gate mean:",
    initial_gate_mean,
)

print(
    "Training extreme prior:",
    training_extreme_rate,
)

Initial correction max |Δ|: 0.0
Initial EA-TTM vs HP-TTM max |difference|: 0.0
Initial gate mean: 0.047052618116140366
Training extreme prior: 0.04705260506957785


## Parameter audit

In [13]:
parameter_records: list[dict[str, Any]] = []

for parameter_name, parameter in adapter.named_parameters():
    parameter_records.append(
        {
            "parameter_name": parameter_name,
            "shape": list(
                parameter.shape
            ),
            "num_parameters": int(
                parameter.numel()
            ),
            "trainable": bool(
                parameter.requires_grad
            ),
        }
    )

parameter_table = pd.DataFrame(
    parameter_records
)

adapter_parameter_count = int(
    parameter_table[
        "num_parameters"
    ].sum()
)

adapter_trainable_parameter_count = int(
    parameter_table.loc[
        parameter_table["trainable"],
        "num_parameters",
    ].sum()
)

assert (
    adapter_parameter_count
    == adapter_trainable_parameter_count
)

assert adapter_parameter_count < 5_000

print(
    "EA-TTM adapter trainable parameters:",
    adapter_trainable_parameter_count,
)

EA-TTM adapter trainable parameters: 2466


## Persist target and architecture artifacts

In [14]:
TRAIN_EXTREME_LABELS_PATH = (
    TARGET_DIR
    / "train_extreme_labels.npy"
)

VALIDATION_EXTREME_LABELS_PATH = (
    TARGET_DIR
    / "validation_extreme_labels.npy"
)

TRAIN_SHOCK_SCORES_PATH = (
    TARGET_DIR
    / "train_shock_scores.npy"
)

VALIDATION_SHOCK_SCORES_PATH = (
    TARGET_DIR
    / "validation_shock_scores.npy"
)

TRAIN_RESIDUAL_TARGETS_PATH = (
    TARGET_DIR
    / "train_residual_targets.npy"
)

VALIDATION_RESIDUAL_TARGETS_PATH = (
    TARGET_DIR
    / "validation_residual_targets.npy"
)

np.save(
    TRAIN_EXTREME_LABELS_PATH,
    training_extreme_labels,
)

np.save(
    VALIDATION_EXTREME_LABELS_PATH,
    validation_extreme_labels,
)

np.save(
    TRAIN_SHOCK_SCORES_PATH,
    training_shock_scores,
)

np.save(
    VALIDATION_SHOCK_SCORES_PATH,
    validation_shock_scores,
)

np.save(
    TRAIN_RESIDUAL_TARGETS_PATH,
    training_residual_targets,
)

np.save(
    VALIDATION_RESIDUAL_TARGETS_PATH,
    validation_residual_targets,
)

In [15]:
ADAPTER_STATE_PATH = (
    MODEL_DIR
    / "adapter_initial_state.pt"
)

ADAPTER_CONFIG_PATH = (
    MODEL_DIR
    / "adapter_config.json"
)

EXTREME_CONTRACT_PATH = (
    METADATA_DIR
    / "07_EXTREME_TARGET_CONTRACT.json"
)

ARCHITECTURE_CONTRACT_PATH = (
    METADATA_DIR
    / "07_EA_TTM_ARCHITECTURE_CONTRACT.json"
)

PARAMETER_TABLE_PATH = (
    RESULTS_TABLE_DIR
    / "07_ADAPTER_PARAMETER_INVENTORY.csv"
)

EXTREME_SUMMARY_PATH = (
    RESULTS_TABLE_DIR
    / "07_EXTREME_TARGET_SUMMARY.csv"
)

RUN_SUMMARY_PATH = (
    METADATA_DIR
    / "07_RUN_SUMMARY.json"
)

torch.save(
    adapter.state_dict(),
    ADAPTER_STATE_PATH,
)

adapter_config = {
    "input_dim": 141,
    "recent_feature_dim": 5,
    "decoder_feature_dim": 128,
    "future_covariate_dim": 8,
    "encoder_width": 16,
    "correction_hidden_width": 8,
    "gate_activation": "sigmoid",
    "encoder_activation": "GELU",
    "encoder_normalization": "LayerNorm",
    "correction_activation": "GELU",
    "correction_output_activation": None,
    "correction_final_layer_zero_initialized": True,
    "gate_weight_zero_initialized": True,
    "gate_bias_initialized_from_training_extreme_prior": True,
    "trainable_parameter_count": adapter_trainable_parameter_count,
}

ADAPTER_CONFIG_PATH.write_text(
    json.dumps(
        adapter_config,
        indent=2,
    ),
    encoding="utf-8",
)

parameter_table.to_csv(
    PARAMETER_TABLE_PATH,
    index=False,
)

extreme_summary = pd.DataFrame(
    [
        {
            "split": "training",
            "threshold": extreme_threshold,
            "extreme_rate": training_extreme_rate,
            "positive_extreme_rate": training_positive_extreme_rate,
            "negative_extreme_rate": training_negative_extreme_rate,
        },
        {
            "split": "validation",
            "threshold": extreme_threshold,
            "extreme_rate": validation_extreme_rate,
            "positive_extreme_rate": validation_positive_extreme_rate,
            "negative_extreme_rate": validation_negative_extreme_rate,
        },
    ]
)

extreme_summary.to_csv(
    EXTREME_SUMMARY_PATH,
    index=False,
)

In [16]:
extreme_contract = {
    "status": "EXTREME_TARGET_CONTRACT_LOCKED",
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "definition": (
        "abs(target - recent_28d_median) "
        "/ max(1.4826 * recent_28d_MAD, 1.0)"
    ),
    "threshold_quantile": EXTREME_QUANTILE,
    "threshold_fit_scope": "training_only",
    "extreme_threshold": extreme_threshold,
    "comparison_operator": ">",
    "training_extreme_rate": training_extreme_rate,
    "validation_extreme_rate": validation_extreme_rate,
    "training_positive_extreme_rate": training_positive_extreme_rate,
    "training_negative_extreme_rate": training_negative_extreme_rate,
    "validation_positive_extreme_rate": validation_positive_extreme_rate,
    "validation_negative_extreme_rate": validation_negative_extreme_rate,
    "robust_scale_floor": ROBUST_SCALE_FLOOR,
    "locked_final_test_used": False,
}

extreme_payload = json.dumps(
    extreme_contract,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

extreme_contract_sha256 = hashlib.sha256(
    extreme_payload
).hexdigest()

extreme_contract[
    "extreme_contract_sha256"
] = extreme_contract_sha256

EXTREME_CONTRACT_PATH.write_text(
    json.dumps(
        extreme_contract,
        indent=2,
    ),
    encoding="utf-8",
)

architecture_contract = {
    "status": "EA_TTM_ARCHITECTURE_CONTRACT_LOCKED",
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "feature_contract_sha256": FEATURE_CONTRACT_SHA256,
    "crossfeed_contract_sha256": CROSSFEED_CONTRACT_SHA256,
    "cache_contract_sha256": CACHE_CONTRACT_SHA256,
    "extreme_contract_sha256": extreme_contract_sha256,
    "base_model": "frozen_head_probed_ttm",
    "input_dimension": 141,
    "encoder_width": 16,
    "correction_hidden_width": 8,
    "adapter_trainable_parameters": adapter_trainable_parameter_count,
    "equation": (
        "ea_forecast = base_forecast "
        "+ sigmoid(gate_logits) * residual_correction"
    ),
    "identity_initialization_verified": (
        initial_forecast_max_abs_difference == 0.0
    ),
    "initial_gate_prior_verified": bool(
        np.isclose(
            initial_gate_mean,
            training_extreme_rate,
            rtol=1e-5,
            atol=1e-6,
        )
    ),
}

architecture_payload = json.dumps(
    architecture_contract,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

architecture_contract_sha256 = hashlib.sha256(
    architecture_payload
).hexdigest()

architecture_contract[
    "architecture_contract_sha256"
] = architecture_contract_sha256

ARCHITECTURE_CONTRACT_PATH.write_text(
    json.dumps(
        architecture_contract,
        indent=2,
    ),
    encoding="utf-8",
)

1053

In [17]:
run_summary = {
    "status": "PHASE_07_COMPLETE",
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "feature_contract_sha256": FEATURE_CONTRACT_SHA256,
    "crossfeed_contract_sha256": CROSSFEED_CONTRACT_SHA256,
    "extreme_contract_sha256": extreme_contract_sha256,
    "architecture_contract_sha256": architecture_contract_sha256,
    "extreme_threshold": extreme_threshold,
    "training_extreme_rate": training_extreme_rate,
    "validation_extreme_rate": validation_extreme_rate,
    "training_positive_extreme_rate": training_positive_extreme_rate,
    "training_negative_extreme_rate": training_negative_extreme_rate,
    "validation_positive_extreme_rate": validation_positive_extreme_rate,
    "validation_negative_extreme_rate": validation_negative_extreme_rate,
    "adapter_input_dimension": 141,
    "encoder_width": 16,
    "correction_hidden_width": 8,
    "adapter_trainable_parameters": adapter_trainable_parameter_count,
    "identity_initialization_max_abs_difference": initial_forecast_max_abs_difference,
    "initial_gate_mean": initial_gate_mean,
    "locked_final_test_used": False,
}

RUN_SUMMARY_PATH.write_text(
    json.dumps(
        run_summary,
        indent=2,
    ),
    encoding="utf-8",
)

1226

In [18]:
status_path = PROJECT_ROOT / "PROJECT_STATUS.md"

status_lines = [
    "# EA-TTM Project Status",
    "",
    "## Current stage",
    "Phase 07 — Extreme target and EA-TTM architecture locked.",
    "",
    "## Extreme Gate target",
    f"- Training-only shock threshold: {extreme_threshold:.8f}",
    f"- Training extreme rate: {training_extreme_rate:.6f}",
    f"- Validation extreme rate: {validation_extreme_rate:.6f}",
    "",
    "## Adapter architecture",
    "- Input dimension: 141",
    "- Shock Encoder: 141 -> 16 + GELU + LayerNorm",
    "- Extreme Gate: 16 -> 1 -> sigmoid",
    "- Correction Head: 16 -> 8 -> 1",
    f"- Trainable adapter parameters: {adapter_trainable_parameter_count:,}",
    "",
    "## Initialization",
    "- Correction output initialized exactly to zero",
    "- EA-TTM therefore starts exactly equal to HP-TTM",
    "- Gate initialized to empirical training extreme prior",
    "",
    f"- Extreme contract SHA-256: `{extreme_contract_sha256}`",
    f"- Architecture contract SHA-256: `{architecture_contract_sha256}`",
    "",
    "## Next phase",
    (
        "Train the EA-TTM adapter with the HP-TTM frozen, "
        "using gate supervision and tail-aware residual forecasting."
    ),
]

status_path.write_text(
    "\n".join(status_lines),
    encoding="utf-8",
)

928

## Phase validation

Phase 07 passes only if:

$$
\begin{aligned}
&\text{extreme threshold fitted on training only}
\\
&\land\
\dim(\mathbf u_h)=141
\\
&\land\
\mathbf r_h\in\mathbb{R}^{16}
\\
&\land\
g_h\in[0,1]
\\
&\land\
\Delta_h\in\mathbb{R}
\\
&\land\
\hat y_{\mathrm{EA},h}^{(0)}
=
\hat y_{\mathrm{HP},h}
\\
&\land\
\text{locked final-test demand remains unused}.
\end{aligned}
$$

In [19]:
phase_validation = {
    "extreme_threshold_finite": bool(
        np.isfinite(extreme_threshold)
    ),
    "training_extreme_labels_valid": (
        training_extreme_labels.shape
        == (121_960, 28)
    ),
    "validation_extreme_labels_valid": (
        validation_extreme_labels.shape
        == (30_490, 28)
    ),
    "adapter_input_shape_valid": (
        tuple(
            smoke_output[
                "adapter_input"
            ].shape
        )
        == (64, 28, 141)
    ),
    "shock_representation_shape_valid": (
        tuple(
            smoke_output[
                "shock_representation"
            ].shape
        )
        == (64, 28, 16)
    ),
    "gate_range_valid": bool(
        (
            smoke_output[
                "gate_probability"
            ] >= 0.0
        ).all()
        and (
            smoke_output[
                "gate_probability"
            ] <= 1.0
        ).all()
    ),
    "identity_initialization_verified": (
        initial_forecast_max_abs_difference == 0.0
    ),
    "adapter_parameter_budget_valid": (
        adapter_trainable_parameter_count < 5_000
    ),
    "locked_test_unused": (
        "d_1914"
        not in development_columns
    ),
    "extreme_contract_saved": (
        EXTREME_CONTRACT_PATH.exists()
    ),
    "architecture_contract_saved": (
        ARCHITECTURE_CONTRACT_PATH.exists()
    ),
    "run_summary_saved": (
        RUN_SUMMARY_PATH.exists()
    ),
}

assert all(
    phase_validation.values()
), (
    f"Phase 07 validation failed: "
    f"{phase_validation}"
)

phase_validation

{'extreme_threshold_finite': True,
 'training_extreme_labels_valid': True,
 'validation_extreme_labels_valid': True,
 'adapter_input_shape_valid': True,
 'shock_representation_shape_valid': True,
 'gate_range_valid': True,
 'identity_initialization_verified': True,
 'adapter_parameter_budget_valid': True,
 'locked_test_unused': True,
 'extreme_contract_saved': True,
 'architecture_contract_saved': True,
 'run_summary_saved': True}

## Return block

The final block is the **only successful-run output that needs to be returned to ChatGPT**.

In [20]:
return_block = {
    "status": "PHASE_07_COMPLETE",
    "dataset_sha256": DATASET_SHA256,
    "split_sha256": SPLIT_SHA256,
    "feature_contract_sha256": FEATURE_CONTRACT_SHA256,
    "crossfeed_contract_sha256": CROSSFEED_CONTRACT_SHA256,
    "extreme_contract_sha256": extreme_contract_sha256,
    "architecture_contract_sha256": architecture_contract_sha256,
    "extreme_definition": (
        "abs(target - recent_28d_median) "
        "/ max(1.4826 * recent_28d_MAD, 1.0)"
    ),
    "extreme_quantile": EXTREME_QUANTILE,
    "extreme_threshold": extreme_threshold,
    "extreme_rates": {
        "training": training_extreme_rate,
        "validation": validation_extreme_rate,
        "training_positive": training_positive_extreme_rate,
        "training_negative": training_negative_extreme_rate,
        "validation_positive": validation_positive_extreme_rate,
        "validation_negative": validation_negative_extreme_rate,
    },
    "adapter_architecture": {
        "input_dimension": 141,
        "shock_encoder": "Linear(141,16) -> GELU -> LayerNorm(16)",
        "extreme_gate": "Linear(16,1) -> sigmoid",
        "correction_head": "Linear(16,8) -> GELU -> Linear(8,1)",
        "trainable_parameters": adapter_trainable_parameter_count,
    },
    "initialization_audit": {
        "initial_gate_mean": initial_gate_mean,
        "training_extreme_prior": training_extreme_rate,
        "initial_correction_max_abs": initial_correction_max_abs,
        "initial_ea_vs_hp_max_abs_difference": initial_forecast_max_abs_difference,
    },
    "target_shapes": {
        "training_extreme_labels": list(
            training_extreme_labels.shape
        ),
        "validation_extreme_labels": list(
            validation_extreme_labels.shape
        ),
        "training_residual_targets": list(
            training_residual_targets.shape
        ),
        "validation_residual_targets": list(
            validation_residual_targets.shape
        ),
    },
    "locked_final_test_used": False,
    "artifacts": {
        "extreme_contract": str(
            EXTREME_CONTRACT_PATH
        ),
        "architecture_contract": str(
            ARCHITECTURE_CONTRACT_PATH
        ),
        "adapter_initial_state": str(
            ADAPTER_STATE_PATH
        ),
        "adapter_config": str(
            ADAPTER_CONFIG_PATH
        ),
        "adapter_parameter_inventory": str(
            PARAMETER_TABLE_PATH
        ),
        "extreme_summary": str(
            EXTREME_SUMMARY_PATH
        ),
        "train_extreme_labels": str(
            TRAIN_EXTREME_LABELS_PATH
        ),
        "validation_extreme_labels": str(
            VALIDATION_EXTREME_LABELS_PATH
        ),
        "train_shock_scores": str(
            TRAIN_SHOCK_SCORES_PATH
        ),
        "validation_shock_scores": str(
            VALIDATION_SHOCK_SCORES_PATH
        ),
        "train_residual_targets": str(
            TRAIN_RESIDUAL_TARGETS_PATH
        ),
        "validation_residual_targets": str(
            VALIDATION_RESIDUAL_TARGETS_PATH
        ),
        "run_summary": str(
            RUN_SUMMARY_PATH
        ),
        "project_status": str(
            status_path
        ),
    },
}

print("PHASE 07 RETURN BLOCK")
print(
    json.dumps(
        return_block,
        indent=2,
    )
)

PHASE 07 RETURN BLOCK
{
  "status": "PHASE_07_COMPLETE",
  "dataset_sha256": "83962ce8bbcb0757215d045f81cac3a7cf4465943334ffa168af326bedc5ed59",
  "split_sha256": "5ff2e30f43a832bb9e513153bb15daa4e1e67c871b43586f2f2f11ef51396447",
  "feature_contract_sha256": "fa0abc3b4c9bf645b24e49b8f8f8582057bf3a37e7bf8fd54c374933213e01fb",
  "crossfeed_contract_sha256": "064448a10cc1fe2b3e9744a94448f83865a3f8ddbec2e315bfcf09e94b93ffb2",
  "extreme_contract_sha256": "dd5506baec503de0407936062ccc764c006ff61a481798d143fa9a31a19d33b9",
  "architecture_contract_sha256": "b1062c1a40bc6ec33111f40e309961710ccd96af56ec07fac9db40c9c3b1ab35",
  "extreme_definition": "abs(target - recent_28d_median) / max(1.4826 * recent_28d_MAD, 1.0)",
  "extreme_quantile": 0.95,
  "extreme_threshold": 2.0,
  "extreme_rates": {
    "training": 0.04705260506957785,
    "validation": 0.0572998172702994,
    "training_positive": 0.043022302394227614,
    "training_negative": 0.004030302675350232,
    "validation_positive": 0.0523